# M2 Gap — ARIMA Univariate Baseline

**Issue:** #54  
**Owner:** Mitchel  
**Reviewer:** Nelson  
**Branch:** `artifact/m2-arima-baseline`

## Objective

Implement the univariate ARIMA statistical baseline required by the project proposal and compare it directly against the shared Random Walk benchmark used in the VAR baselines.

The model will use the stationary Canadian 10Y–2Y yield-spread series and will follow the same evaluation framework used in Issues #28 and #29 to preserve direct comparability across models.

## Methodology

- Target: `yield_spread_10y_2y`
- Stationarity treatment: first-differenced spread based on the Round 2 ADF findings
- ARIMA order selection: automated search minimizing AIC/BIC
- Forecast horizons: 1, 5, and 20 observations
- Expanding-window evaluation
- Initial training sample: 500 observations
- Evaluation step: 5 observations
- Benchmark: Random Walk
- Metrics: RMSE and MAE
- Statistical comparison: Diebold-Mariano test using the same methodology as #28/#29

## Comparability requirement

The ARIMA baseline must reproduce from the same processed data and evaluation window used by the corrected Round 3 VAR baselines so that ARIMA, VAR-BIC, VAR-AIC, and the Random Walk can be compared directly.

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

PROJECT_ROOT = Path.cwd().resolve().parents[1]
PROCESSED = PROJECT_ROOT / "data" / "processed"

TARGET = "yield_spread_10y_2y"
HORIZONS = [1, 5, 20]

MIN_TRAIN = 500
STEP = 5

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED)
print("Target:", TARGET)
print("Horizons:", HORIZONS)

Project root: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5
Processed data: C:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\data\processed
Target: yield_spread_10y_2y
Horizons: [1, 5, 20]


In [7]:
# Load Bank of Canada processed data
boc = pd.read_csv(
    PROCESSED / "bank_of_canada_data.csv",
    parse_dates=["date"]
)

spread_levels = (
    boc[["date", TARGET]]
    .dropna(subset=[TARGET])
    .sort_values("date")
    .reset_index(drop=True)
)

spread_diff = spread_levels.copy()
spread_diff[TARGET] = spread_diff[TARGET].diff()

spread_diff = (
    spread_diff
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

print("Level observations:", len(spread_levels))
print("Differenced observations:", len(spread_diff))
print(
    "Date range:",
    spread_diff["date"].min(),
    "->",
    spread_diff["date"].max()
)

spread_diff.head()

Level observations: 4368
Differenced observations: 4367
Date range: 2009-01-05 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y
0,2009-01-05,0.06
1,2009-01-06,-0.01
2,2009-01-07,0.06
3,2009-01-08,-0.06
4,2009-01-09,-0.01


In [8]:
# Build the same common evaluation sample used by #28/#29

fred = pd.read_csv(
    PROCESSED / "fred_rates.csv",
    parse_dates=["date"]
)

cpi = pd.read_csv(
    PROCESSED / "statcan_cpi.csv",
    parse_dates=["reference_month", "release_date"]
)

# Same five-variable alignment used by the VAR baselines
daily = (
    boc.merge(fred, on="date", how="inner")
       .sort_values("date")
       .reset_index(drop=True)
)

# Compute CPI YoY before expanding monthly releases to daily frequency
cpi_monthly = cpi.sort_values("reference_month").copy()
cpi_monthly["cpi_yoy"] = (
    cpi_monthly["cpi_all_items"].pct_change(12) * 100
)

cpi_daily = (
    cpi_monthly[["release_date", "cpi_yoy"]]
    .rename(columns={"release_date": "date"})
    .sort_values("date")
)

daily = (
    daily.merge(cpi_daily, on="date", how="left")
         .sort_values("date")
         .reset_index(drop=True)
)

daily["cpi_yoy"] = daily["cpi_yoy"].ffill()

COMMON_FEATURES = [
    "yield_spread_10y_2y",
    "overnight_rate",
    "us_treasury_10y",
    "fed_funds_rate",
    "cpi_yoy",
]

# Drop incomplete level observations BEFORE differencing,
# exactly as in the corrected VAR baseline
common_levels = (
    daily[["date"] + COMMON_FEATURES]
    .dropna(subset=COMMON_FEATURES)
    .sort_values("date")
    .reset_index(drop=True)
)

# ARIMA remains univariate: retain only the spread after date alignment
arima_levels = common_levels[["date", TARGET]].copy()

arima_diff = arima_levels.copy()
arima_diff[TARGET] = arima_diff[TARGET].diff()

arima_diff = (
    arima_diff
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

print("Common level observations:", len(arima_levels))
print("ARIMA differenced observations:", len(arima_diff))
print(
    "ARIMA modeling date range:",
    arima_diff["date"].min(),
    "->",
    arima_diff["date"].max()
)

Common level observations: 4010
ARIMA differenced observations: 4009
ARIMA modeling date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


In [9]:
# Automated ARIMA order search on the stationary spread series

stationary_spread = arima_diff[TARGET].astype(float)

MAX_P = 5
MAX_Q = 5

order_search_results = []

for p in range(MAX_P + 1):
    for q in range(MAX_Q + 1):
        order = (p, 0, q)

        try:
            fitted = ARIMA(
                stationary_spread,
                order=order,
                trend="c"
            ).fit()

            order_search_results.append({
                "p": p,
                "d": 0,
                "q": q,
                "aic": fitted.aic,
                "bic": fitted.bic,
            })

        except Exception as exc:
            print(f"Skipped ARIMA{order}: {exc}")

order_search_df = (
    pd.DataFrame(order_search_results)
    .sort_values("aic")
    .reset_index(drop=True)
)

best_aic_row = order_search_df.loc[
    order_search_df["aic"].idxmin()
]

best_bic_row = order_search_df.loc[
    order_search_df["bic"].idxmin()
]

BEST_AIC_ORDER = (
    int(best_aic_row["p"]),
    0,
    int(best_aic_row["q"])
)

BEST_BIC_ORDER = (
    int(best_bic_row["p"]),
    0,
    int(best_bic_row["q"])
)

print("Best order by AIC:", BEST_AIC_ORDER)
print("AIC:", round(best_aic_row["aic"], 3))

print("\nBest order by BIC:", BEST_BIC_ORDER)
print("BIC:", round(best_bic_row["bic"], 3))

print("\nTop 10 models by AIC:")
display(order_search_df.head(10))

c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.ve

Best order by AIC: (0, 0, 5)
AIC: -16793.3

Best order by BIC: (0, 0, 0)
BIC: -16775.232

Top 10 models by AIC:


c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,p,d,q,aic,bic
0,0,0,5,-16793.300307,-16749.226227
1,5,0,0,-16793.112084,-16749.038004
2,2,0,4,-16791.386598,-16741.016222
3,1,0,5,-16791.167651,-16740.797274
4,5,0,1,-16791.118841,-16740.748464
5,3,0,3,-16790.119989,-16739.749612
6,5,0,3,-16789.922462,-16726.959491
7,2,0,5,-16789.450147,-16732.783473
8,3,0,5,-16788.835117,-16725.872146
9,3,0,4,-16788.782796,-16732.116122


In [10]:
# Validate convergence of the AIC- and BIC-selected specifications

selected_models = {}

for criterion, order in {
    "AIC": BEST_AIC_ORDER,
    "BIC": BEST_BIC_ORDER,
}.items():

    fit = ARIMA(
        stationary_spread,
        order=order,
        trend="c"
    ).fit()

    selected_models[criterion] = fit

    converged = fit.mle_retvals.get("converged", True)

    print(f"{criterion}-selected model: ARIMA{order}")
    print(f"Converged: {converged}")
    print(f"AIC: {fit.aic:.3f}")
    print(f"BIC: {fit.bic:.3f}")
    print("-" * 40)

AIC-selected model: ARIMA(0, 0, 5)
Converged: True
AIC: -16793.300
BIC: -16749.226
----------------------------------------
BIC-selected model: ARIMA(0, 0, 0)
Converged: True
AIC: -16787.825
BIC: -16775.232
----------------------------------------


In [11]:
# Align spread levels exactly with the differenced modeling dates
aligned_arima_levels = (
    arima_levels[
        arima_levels["date"].isin(arima_diff["date"])
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

assert len(aligned_arima_levels) == len(arima_diff)
assert aligned_arima_levels["date"].equals(arima_diff["date"])

print("Aligned ARIMA observations:", len(aligned_arima_levels))
print(
    "Aligned date range:",
    aligned_arima_levels["date"].min(),
    "->",
    aligned_arima_levels["date"].max()
)

Aligned ARIMA observations: 4009
Aligned date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


In [12]:
ARIMA_ORDERS = {
    "arima_aic": BEST_AIC_ORDER,
    "arima_bic": BEST_BIC_ORDER,
}

results = []

for origin in range(MIN_TRAIN, len(arima_diff) - max(HORIZONS), STEP):
    train_diff = arima_diff.iloc[:origin][TARGET].astype(float)

    # Last observed level at forecast origin
    last_level = aligned_arima_levels.loc[origin - 1, TARGET]

    fitted_models = {}

    for model_name, order in ARIMA_ORDERS.items():
        fitted_models[model_name] = ARIMA(
            train_diff,
            order=order,
            trend="c"
        ).fit()

    for h in HORIZONS:
        actual_level = aligned_arima_levels.loc[origin + h - 1, TARGET]

        # Shared Random Walk benchmark
        naive_forecast = last_level

        row = {
            "origin_date": aligned_arima_levels.loc[origin - 1, "date"],
            "horizon": h,
            "actual": actual_level,
            "naive": naive_forecast,
        }

        for model_name, fit in fitted_models.items():
            forecast_diff = np.asarray(
                fit.forecast(steps=h),
                dtype=float
            )

            cumulative_change = forecast_diff.sum()
            row[model_name] = last_level + cumulative_change

        results.append(row)

results_df = pd.DataFrame(results)

print("Forecast rows:", len(results_df))
print(results_df.groupby("horizon").size())

results_df.head()

c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\ba

Forecast rows: 2094
horizon
1     698
5     698
20    698
dtype: int64


,origin_date,horizon,actual,naive,arima_aic,arima_bic
0,2012-03-02,1,0.87,0.86,0.859470,0.857475
1,2012-03-02,5,0.84,0.86,0.867148,0.847375
2,2012-03-02,20,0.91,0.86,0.830098,0.809500
3,2012-03-09,1,0.82,0.84,0.840653,0.837460
4,2012-03-09,5,0.96,0.84,0.835820,0.827302


In [13]:
print("Total forecast rows:", len(results_df))

print("\nMissing forecasts:")
print(
    results_df[
        ["arima_aic", "arima_bic", "naive", "actual"]
    ].isna().sum()
)

print("\nInfinite forecasts:")
print(
    np.isinf(
        results_df[
            ["arima_aic", "arima_bic", "naive", "actual"]
        ]
    ).sum()
)

print("\nForecast ranges:")
print(
    results_df[
        ["actual", "naive", "arima_aic", "arima_bic"]
    ].agg(["min", "max"])
)

Total forecast rows: 2094

Missing forecasts:
arima_aic    0
arima_bic    0
naive        0
actual       0
dtype: int64

Infinite forecasts:
arima_aic    0
arima_bic    0
naive        0
actual       0
dtype: int64

Forecast ranges:
     actual  naive  arima_aic  arima_bic
min   -1.31  -1.31  -1.323821  -1.331046
max    1.61   1.61   1.601343   1.609458


In [14]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]

    row = {
        "horizon": h
    }

    for model_name in ["arima_aic", "arima_bic", "naive"]:
        row[f"{model_name}_rmse"] = np.sqrt(
            mean_squared_error(
                subset["actual"],
                subset[model_name]
            )
        )

        row[f"{model_name}_mae"] = mean_absolute_error(
            subset["actual"],
            subset[model_name]
        )

    # Improvement relative to Random Walk
    for model_name in ["arima_aic", "arima_bic"]:
        row[f"{model_name}_rmse_improvement_pct"] = (
            (row["naive_rmse"] - row[f"{model_name}_rmse"])
            / row["naive_rmse"]
            * 100
        )

        row[f"{model_name}_mae_improvement_pct"] = (
            (row["naive_mae"] - row[f"{model_name}_mae"])
            / row["naive_mae"]
            * 100
        )

    metrics.append(row)

metrics_df = pd.DataFrame(metrics)

metrics_df.round(6)

,horizon,arima_aic_rmse,arima_aic_mae,arima_bic_rmse,arima_bic_mae,naive_rmse,naive_mae,arima_aic_rmse_improvement_pct,arima_aic_mae_improvement_pct,arima_bic_rmse_improvement_pct,arima_bic_mae_improvement_pct
0,1,0.030423,0.022638,0.030261,0.022415,0.030238,0.022321,-0.611751,-1.419297,-0.075047,-0.420491
1,5,0.064215,0.048804,0.063650,0.048552,0.063416,0.048438,-1.258888,-0.754159,-0.368560,-0.233794
2,20,0.133083,0.100771,0.132461,0.100234,0.130718,0.098825,-1.809721,-1.968698,-1.333798,-1.425835


In [15]:
# Diebold-Mariano tests: ARIMA-AIC vs Naive and ARIMA-BIC vs Naive

from dieboldmariano import dm_test

dm_results = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h].copy()

    actual = subset["actual"].to_numpy()
    naive = subset["naive"].to_numpy()

    for model_name in ["arima_aic", "arima_bic"]:
        model_fcst = subset[model_name].to_numpy()

        # Squared-error loss
        dm_stat_sq, p_sq = dm_test(
            actual,
            model_fcst,
            naive,
            h=h,
            one_sided=False,
            harvey_correction=True,
            variance_estimator="bartlett"
        )

        # Absolute-error loss
        dm_stat_abs, p_abs = dm_test(
            actual,
            model_fcst,
            naive,
            h=h,
            loss=lambda u, v: abs(u - v),
            one_sided=False,
            harvey_correction=True,
            variance_estimator="bartlett"
        )

        dm_results.append({
            "model": model_name,
            "horizon_days": h,
            "dm_stat_squared_loss": dm_stat_sq,
            "dm_p_value_squared_loss": p_sq,
            "dm_stat_absolute_loss": dm_stat_abs,
            "dm_p_value_absolute_loss": p_abs,
        })

dm_results_df = pd.DataFrame(dm_results)

dm_results_df.round(4)

,model,horizon_days,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss
0,arima_aic,1,1.5890,0.1125,2.9524,0.0033
1,arima_bic,1,0.6651,0.5062,2.5088,0.0123
2,arima_aic,5,2.2321,0.0259,1.0522,0.2931
3,arima_bic,5,1.3088,0.1910,0.6081,0.5433
4,arima_aic,20,1.3842,0.1667,1.2536,0.2104
5,arima_bic,20,1.1725,0.2414,1.0231,0.3066


In [16]:
# Convergence diagnostic across expanding windows

convergence_records = []

for origin in range(MIN_TRAIN, len(arima_diff) - max(HORIZONS), STEP):
    train_diff = arima_diff.iloc[:origin][TARGET].astype(float)

    for model_name, order in ARIMA_ORDERS.items():
        fit = ARIMA(
            train_diff,
            order=order,
            trend="c"
        ).fit()

        convergence_records.append({
            "origin": origin,
            "origin_date": aligned_arima_levels.loc[origin - 1, "date"],
            "model": model_name,
            "order": str(order),
            "converged": bool(fit.mle_retvals.get("converged", True)),
        })

convergence_df = pd.DataFrame(convergence_records)

convergence_summary = (
    convergence_df
    .groupby(["model", "order"])["converged"]
    .agg(
        total_fits="count",
        converged_fits="sum"
    )
    .reset_index()
)

convergence_summary["non_converged_fits"] = (
    convergence_summary["total_fits"]
    - convergence_summary["converged_fits"]
)

convergence_summary["convergence_rate_pct"] = (
    convergence_summary["converged_fits"]
    / convergence_summary["total_fits"]
    * 100
)

convergence_summary

c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\ba

,model,order,total_fits,converged_fits,non_converged_fits,convergence_rate_pct
0,arima_aic,"(0, 0, 5)",698,651,47,93.266476
1,arima_bic,"(0, 0, 0)",698,669,29,95.845272


In [17]:
# Retry only non-converged expanding-window fits with a higher iteration limit

retry_records = []

failed_fits = convergence_df[
    convergence_df["converged"] == False
].copy()

for _, failed in failed_fits.iterrows():

    origin = int(failed["origin"])
    model_name = failed["model"]
    order = ARIMA_ORDERS[model_name]

    train_diff = arima_diff.iloc[:origin][TARGET].astype(float)

    retry_fit = ARIMA(
        train_diff,
        order=order,
        trend="c"
    ).fit(
        method_kwargs={"maxiter": 500}
    )

    retry_records.append({
        "origin": origin,
        "origin_date": aligned_arima_levels.loc[origin - 1, "date"],
        "model": model_name,
        "order": str(order),
        "retry_converged": bool(
            retry_fit.mle_retvals.get("converged", True)
        ),
    })

retry_df = pd.DataFrame(retry_records)

retry_summary = (
    retry_df
    .groupby("model")["retry_converged"]
    .agg(
        retried="count",
        converged_after_retry="sum"
    )
    .reset_index()
)

retry_summary["still_not_converged"] = (
    retry_summary["retried"]
    - retry_summary["converged_after_retry"]
)

retry_summary

c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Mitchel\OneDrive - GUSCanada\Documents\GitHub\DAMO-699-Capstone-project-GRP5\.venv\Lib\site-packages\statsmodels\ba

,model,retried,converged_after_retry,still_not_converged
0,arima_aic,47,0,47
1,arima_bic,29,0,29


### Expanding-window convergence diagnostic

Because ARIMA models are re-estimated at every expanding-window forecast origin, convergence was checked for each fitted specification.

| Model | Order | Total fits | Converged | Non-converged | Convergence rate |
|---|---|---:|---:|---:|---:|
| ARIMA-AIC | (0, 0, 5) | 698 | 651 | 47 | 93.27% |
| ARIMA-BIC | (0, 0, 0) | 698 | 669 | 29 | 95.85% |

A retry of all non-converged fits with a higher maximum iteration limit (`maxiter=500`) did not change their optimizer convergence status.

Despite these optimizer warnings, all forecast origins produced finite predictions with no missing or infinite values, and forecast ranges remained consistent with the observed yield-spread range. The convergence limitation is therefore retained and documented rather than changing model specifications solely to force optimizer convergence.

In [18]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_df.to_csv(
    OUTPUT_DIR / "r3_arima_vs_naive.csv",
    index=False
)

results_df.to_csv(
    OUTPUT_DIR / "r3_arima_forecasts.csv",
    index=False
)

dm_results_df.to_csv(
    OUTPUT_DIR / "r3_arima_diebold_mariano.csv",
    index=False
)

convergence_summary.to_csv(
    OUTPUT_DIR / "r3_arima_convergence.csv",
    index=False
)

print("Saved ARIMA outputs:")
print("- r3_arima_vs_naive.csv")
print("- r3_arima_forecasts.csv")
print("- r3_arima_diebold_mariano.csv")
print("- r3_arima_convergence.csv")

Saved ARIMA outputs:
- r3_arima_vs_naive.csv
- r3_arima_forecasts.csv
- r3_arima_diebold_mariano.csv
- r3_arima_convergence.csv


## ARIMA Baseline Conclusion

The ARIMA baseline was implemented on the stationary first-differenced Canadian 10Y–2Y yield-spread series using the same processed data and evaluation window as the VAR baselines in `#28` and `#29`.

To preserve direct comparability, the ARIMA evaluation used the same common sample:

- **4,010 complete level observations**
- **4,009 differenced modeling observations**
- Expanding-window evaluation
- Initial training sample: **500 observations**
- Evaluation step: **5 observations**
- Forecast horizons: **1, 5, and 20 days**
- Shared benchmark: **Random Walk**

### Order selection

An automated grid search over ARIMA specifications with `p, q ∈ {0, ..., 5}` was performed on the stationary differenced yield-spread series.

The selected specifications were:

- **AIC-selected model:** `ARIMA(0, 0, 5)`
  - AIC = `-16793.300`
  - BIC = `-16749.226`

- **BIC-selected model:** `ARIMA(0, 0, 0)`
  - AIC = `-16787.825`
  - BIC = `-16775.232`

Both selected full-sample specifications converged successfully.

Because the yield spread was first-differenced prior to model estimation, `d = 0` in these ARIMA specifications. The differencing required to achieve stationarity was therefore applied explicitly before fitting the models.

### Forecast performance

| Horizon | ARIMA-AIC RMSE | ARIMA-AIC MAE | ARIMA-BIC RMSE | ARIMA-BIC MAE | Random Walk RMSE | Random Walk MAE |
|---|---:|---:|---:|---:|---:|---:|
| 1 day | 0.030423 | 0.022638 | 0.030261 | 0.022415 | 0.030238 | 0.022321 |
| 5 days | 0.064215 | 0.048804 | 0.063650 | 0.048552 | 0.063416 | 0.048438 |
| 20 days | 0.133083 | 0.100771 | 0.132461 | 0.100234 | 0.130718 | 0.098825 |

Neither ARIMA specification outperformed the Random Walk benchmark across the three forecast horizons.

Relative to the Random Walk:

- **ARIMA-AIC**
  - 1 day: RMSE +0.61%, MAE +1.42%
  - 5 days: RMSE +1.26%, MAE +0.75%
  - 20 days: RMSE +1.81%, MAE +1.97%

- **ARIMA-BIC**
  - 1 day: RMSE +0.08%, MAE +0.42%
  - 5 days: RMSE +0.37%, MAE +0.23%
  - 20 days: RMSE +1.33%, MAE +1.43%

The BIC-selected specification remained closer to the Random Walk than the AIC-selected specification, but still did not improve forecast accuracy.

### Diebold-Mariano significance

The Diebold-Mariano test was applied using horizon-specific truncation lags (`h = 1, 5, 20`) to account for the serial dependence induced by multi-step forecast errors.

For **ARIMA-AIC vs Random Walk**:

- **1 day:** Random Walk significantly better under absolute-error loss (`p = 0.0033`); squared-error difference not significant (`p = 0.1125`).
- **5 days:** Random Walk significantly better under squared-error loss (`p = 0.0259`); absolute-error difference not significant (`p = 0.2931`).
- **20 days:** no statistically significant difference under either squared-error (`p = 0.1667`) or absolute-error (`p = 0.2104`) loss.

For **ARIMA-BIC vs Random Walk**:

- **1 day:** Random Walk significantly better under absolute-error loss (`p = 0.0123`); squared-error difference not significant (`p = 0.5062`).
- **5 days:** no statistically significant difference under either squared-error (`p = 0.1910`) or absolute-error (`p = 0.5433`) loss.
- **20 days:** no statistically significant difference under either squared-error (`p = 0.2414`) or absolute-error (`p = 0.3066`) loss.

Although both ARIMA specifications have slightly higher RMSE and MAE than the Random Walk at all horizons, the Diebold-Mariano results show that these differences are not consistently statistically significant, especially at the 20-day horizon.

### Convergence diagnostic

Because the ARIMA models were re-estimated at every expanding-window forecast origin, optimizer convergence was monitored across all 698 fits per specification.

- `ARIMA(0, 0, 5)`: **651 / 698 converged (93.27%)**
- `ARIMA(0, 0, 0)`: **669 / 698 converged (95.85%)**

Retrying the non-converged fits with a larger optimization limit (`maxiter=500`) did not change their convergence status.

Despite these warnings, all forecast origins produced finite predictions with no missing or infinite values, and forecast ranges remained consistent with the observed yield-spread range. This limitation is therefore documented rather than modifying the selected model specifications solely to force optimizer convergence.

### Overall finding

The ARIMA statistical baseline does not outperform the Random Walk benchmark for Canadian 10Y–2Y yield-spread forecasting under the shared 1-, 5-, and 20-day evaluation framework.

AIC favored a more complex `MA(5)` structure, while BIC favored the parsimonious `ARIMA(0,0,0)` specification. However, neither information-criterion choice produced superior predictive accuracy relative to the naïve benchmark.

These results provide the required univariate statistical baseline for subsequent comparison with VAR/VECM and LSTM models.